# Advanced Statistical Inference -- Variational Inference for Bayesian Logistic Regression

In this notebook, you will learn how to implement the (stochastic) variational inference algorithm.
The gist below will serve you as a refresh on variational inference.
For additional information, please check the lecture notes.

In the general setting, given a probabilistic model with observations $\{\boldsymbol{X},\boldsymbol{y}\}$, model parameters $\boldsymbol{w}$ and likelihood $p(\boldsymbol{y}|\boldsymbol{X}, \boldsymbol{w})$, by introducing an approximate posterior distribution $q_\theta(\boldsymbol{w})$ with parameters $\theta$, the variational lower bound to the log-marginal likelihood is defined as


$$\mathrm{KL}[{q_\theta(\boldsymbol{w})}||{p(\boldsymbol{w}|\boldsymbol{X},\boldsymbol{y})}] = -\mathbb{E}_{q_{\theta}}\log p(\boldsymbol{y}|\boldsymbol{X}, \boldsymbol{w}) +  \mathrm{KL}[{q_{\theta}(\boldsymbol{w})}||{p(\boldsymbol{w})}] + \log p(\boldsymbol{y}|\boldsymbol{X}) $$


The objective is then to maximize this variational bound (or evidence lower bound):

$$
      \mathcal{L}(\theta) = \underbrace{\mathbb{E}_{q_{\theta}}\log p(\boldsymbol{y}|\boldsymbol{X}, \boldsymbol{w})}_\text{Expected loglikelihood} -  \mathrm{KL}[{q_{\theta}(\boldsymbol{w})}||{p(\boldsymbol{w})}]
$$

The analytic evaluation of the ELBO is generally still untractable due to the presence of the expected loglikelihood under the variational distribution (in the majority of  cases the rightmost KL is tractable).
This is commonly overcome by sampling $N_\mathrm{MC}$ times from $q_\theta$  using the reparameterization trick

$$
    \mathbb{E}_{q_{\theta}}\log p(\boldsymbol{y}|\boldsymbol{X}, \boldsymbol{w}) \approx \dfrac{1}{N_\mathrm{MC}} \sum_{\tilde{\boldsymbol{w}}_i\sim q_\theta} \log p(\boldsymbol{y}|\boldsymbol{X}, \tilde{\boldsymbol{w}}_i)
$$

In [ ]:
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib import rc
from tqdm.notebook import tqdm

dark = True
colab = "google.colab" in str(get_ipython())
preamble = r"""\renewcommand{\familydefault}{\sfdefault}\usepackage{sansmath}
\usepackage{FiraSans}\sansmath\usepackage{amsmath}"""

rc("font", **{"family": "sans-serif", "sans-serif": "DejaVu Sans"})
rc("text", **{"usetex": False, "latex.preamble": preamble})
rc("figure", **{"dpi": 200})
rc(
    "axes",
    **{"spines.right": False, "spines.top": False, "xmargin": 0.0, "ymargin": 0.05},
)

## Library and coding

This lab is heavily built on JAX. You already used JAX during the lab on the previous labs.
Revisit the JAX tutorial to refresh your memory, if needed.

# Setup and data

Similarly to the previous lab, you’re going to implement the VI algorithm described in the lecture for binary classification.

In [ ]:
def plot_data(X, y, ax):
    mask = y == 1
    config = dict(edgecolor="black", linewidth=1, zorder=10)
    ax.scatter(*X[mask].T, label="Class 1", facecolor="tab:blue", **config)
    ax.scatter(*X[~mask].T, label="Class 0", facecolor="tab:orange", **config)


def get_grid(xlim=(-3, 3), ylim=None, N=100):
    if ylim is None:
        ylim = xlim
    x_grid = np.linspace(*xlim, N)
    y_grid = np.linspace(*ylim, N)
    xx, yy = np.meshgrid(x_grid, y_grid)
    X_plot = np.vstack((xx.flatten(), yy.flatten())).T
    return xx, yy, X_plot

In [ ]:
data = np.loadtxt(
    "https://github.com/eurecom-ds/asi-labs/raw/refs/heads/master/lab_week2/binaryclass2.csv",
    delimiter=",",
)
X = data[..., :-1]
y = data[..., -1]

fig, ax = plt.subplots(figsize=[5, 3])
plot_data(X, y, ax)
ax.set_title("Binary classification")
ax.legend()
ax.set_xlim(-6, 6)
ax.set_ylim(-6, 6)
plt.show()

Let's start by defining some probability distributions that we will need in this notebook.
First, the Bernoulli distribution.

**Exercise:**
Complete the following class to compute the logdensity of the Bernoulli distribution. You should already have done this in the previous lab.

In [ ]:
def bernoulli_logdensity(y, p):
    # @@ COMPLETE @@
    # out =
    return out

Now let's move to the Gaussian distribution. Note that the parameter $\sigma^2$ is always expected to be positive while it is possible that the optimisation algorithm attempts to evaluate the log-likelihood in regions of the parameter space where one or more of these parameters are negative, leading to numerical issues.
A commonly-used technique to enforce this condition is to work with a transformed version of parameters using the logarithm transformation.
In particular, define $\psi = \log\sigma^2$.

So remember to take the exponential of $\psi$ if you want to use $\sigma^2$.

The following class implements the Gaussian distribution, with a vector of means and a vector of log-variances.

In [ ]:
from functools import partial
from typing import NamedTuple

ParametrizedDistribution = NamedTuple


class GaussianDiagonal(ParametrizedDistribution):
    mean: jnp.array
    log_var: jnp.array

**Exercise**: Complete the next function to generate *one* sample from a Gaussian distribution using the reparameterization trick. Remember $p(\boldsymbol z) = p(t(\boldsymbol \varepsilon;\boldsymbol \theta))$ where $t(\cdot) = \boldsymbol \mu + \boldsymbol \sigma \odot \boldsymbol \varepsilon$ and $\boldsymbol \varepsilon \sim \mathcal{N}(\boldsymbol 0, \boldsymbol I)$

In [ ]:
def sample_gaussian_diagonal(rng, params: ParametrizedDistribution):
    """
    Sample from a Gaussian distribution with diagonal covariance.

    Args:
        rng: Random number generator key.
        params: A named tuple containing the mean and log variance of the Gaussian.
            mean: Mean of the Gaussian distribution.
            log_var: Log variance of the Gaussian distribution.
    """
    # @@ COMPLETE @@
    # eps =
    # out =
    return out

**Exercise**: Create a Gaussian with two components with $\mathcal{N}(\boldsymbol 0, \boldsymbol 1)$ and get one sample.

In [ ]:
rng = jax.random.PRNGKey(0)
p_params = GaussianDiagonal(jnp.zeros(2), jnp.zeros(2))
print(sample_gaussian_diagonal(rng, p_params))

# 2. KL Divergence

The expression of the KL divergence between multivariate Gaussians $q = \mathcal{N}(\boldsymbol{\mu}_q, \boldsymbol\Sigma_q)$ and $p = \mathcal{N}(\boldsymbol{\mu}_p, \boldsymbol\Sigma_p)$ is as follows:

\begin{equation}
\mathrm{KL}[q || p] =
\frac{1}{2} \mathrm{Tr}(\boldsymbol\Sigma_p^{-1} \boldsymbol\Sigma_q)
+ \frac{1}{2} (\boldsymbol\mu_p - \boldsymbol\mu_q)^{\top} \boldsymbol\Sigma_1^{-1} (\boldsymbol\mu_p - \boldsymbol\mu_q)
- \frac{D}{2}
+ \frac{1}{2} \log\left( \frac{\mathrm{det}\boldsymbol\Sigma_p}{\mathrm{det}\boldsymbol\Sigma_q} \right)
\end{equation}


This formula simplifies when the two Gaussians have diagonal covariance, i.e. $q = \mathcal{N}(\boldsymbol{\mu}_q, \boldsymbol{\sigma}^2_q\mathrm{I})$ and $p = \mathcal{N}(\boldsymbol{\mu}_p, \boldsymbol{\sigma}^2_p\mathrm{I})$,

$$
\mathrm{KL}[q || p] =  \frac{1}{2} \sum\left( \log \frac{\sigma^2_p}{\sigma^2_q} + \frac{\sigma_q^2 + (\mu_q - \mu_p)^2}{\sigma_p^2} - 1 \right)
$$


**Exercise:**
Complete the next function to compute the KL divergence between two multivariate Gaussian distribution with diagonal covariance. *Note:* Since we have parameterized the Gaussian distribution with the log-variance, the formula above can be simplified even further.

In [ ]:
def kl_diag_diag(q_params: GaussianDiagonal, p_params: GaussianDiagonal):
    """
    Compute the KL divergence between two Gaussian distributions with diagonal covariance.

    Args:
        q_params: A named tuple containing the mean and log variance of the first Gaussian.
        p_params: A named tuple containing the mean and log variance of the second Gaussian.
    """
    assert isinstance(q_params, GaussianDiagonal)
    assert isinstance(p_params, GaussianDiagonal)

    # @@ COMPLETE @@
    # kl =
    return kl

**Exercise:**
Create two identical Gaussian distributions and compute the KL divergence using the function. What's the result? Is it what you were expecting?

In [ ]:
# @@ COMPLETE @@
# p_params =
# q_params =

print("KL =", kl_diag_diag(q_params, p_params))

# Model

Now we can move to design the model. We will use a simple logistic regression very similarly to what done in the previous MCMC lab.
This very simple model computes $h(\boldsymbol{x}) = h(\boldsymbol{w}^\top\boldsymbol{x})$, where $h(\cdot)$ is the logistic function.

**Exercise:**
Complete the next two functions to compute (i) the logistic function and (ii) the output of the model, given a particular choice of $\boldsymbol w$.

In [ ]:
def logistic(z):
    """
    Logistic function.
    """

    # @@ COMPLETE @@
    # out =
    return out


def model(w, X):
    """
    Logistic regression model.
    Args:
        w: Model parameters.
        X: Input data.
    """
    # @@ COMPLETE @@
    # out =
    return out

# 4. Variational objective

The objective is to maximize this variational bound:
$$
      \mathcal{L}(\theta) = \underbrace{\mathbb{E}_{q_{\theta}}\log p(\boldsymbol{y}|\boldsymbol{X}, \boldsymbol{w})}_\text{Expected loglikelihood} -\mathrm{KL}[{q_{\theta}(\boldsymbol{w})}||{p(\boldsymbol{w})}]
$$

**Exercise:**
Complete the next cell to compute the ELBO.

In [ ]:
def create_elbo_fn(sample_fn, likelihood_fn, kl_divergence_fn):
    """
    Create a function to compute the ELBO, given the function to sample
    from the posterior, the likelihood function and the KL divergence

    Args:
        sample_fn: Function to sample from the posterior.
        likelihood_fn: Likelihood function.
        kl_divergence_fn: KL divergence function
    """

    @partial(jax.vmap, in_axes=[0, None, None, None])
    def likelihood_sample_fn(rng, q_params, X, y):
        """
        Compute the log-likelihood with one Monte Carlo sample of the posterior
        The function is decorated to vectorized multiple MC sample automatically

        Args:
            rng: Random number generator key.
            q_params: Parameters of the posterior distribution.
            X: Input data.
            y: Labels.
        """
        # @@ COMPLETE @@
        # # Get one sample of w using the sample_fn and the parameters of q
        # w =
        # # Predict the output using the sample before
        # yp =
        # # Compute the likelihood and return it (remember that the data points are independent, so the likelihood is the product of the individual likelihoods, which is the ... of the individual log likelihoods)
        # ll =
        return ll

    def elbo_fn(q_params, p_params, rng, X, y, Nmc=1):
        """
        Computes the ELBO with multiple samples

        Args:
            q_params: Parameters of the posterior distribution.
            p_params: Parameters of the prior distribution.
            rng: Random number generator key.
            X: Input data.
            y: Labels.
            Nmc: Number of Monte Carlo samples to use for the ELBO estimation.
        """
        # Split the random seed in Nmc times
        rng = jax.random.split(rng, Nmc)

        # @@ COMPLETE @@
        # # Compute the values of the elbo
        # likelihood_vals =
        # # Compute the expectation (i.e. take the mean)
        # expected_likelihood =
        # # Compute the KL divergence
        # kl =
        # # Compute the ELBO
        # elbo =

        # # Return the ELBO and its two term (used later for logging)
        return elbo, (expected_likelihood, kl)

    return jax.jit(elbo_fn, static_argnames=("Nmc"))

**Exercise:**
Using the function above, create the function to compute the ELBO.

In [ ]:
elbo_fn = create_elbo_fn(
    sample_fn=sample_gaussian_diagonal,
    kl_divergence_fn=kl_diag_diag,
    likelihood_fn=bernoulli_logdensity,
)

**Exercise:**
Try to compute the variational objective (use 10 Monte Carlo samples).

In [ ]:
# @@ COMPLETE @@
# elbo, (likelihood, kl) =

print("ELBO =", elbo)
print("Likelihood =", likelihood)
print("KL =", kl)

## Analysis of the MC estimate of the ELBO

Ok, now that everything is done and ready we can start to make some analysis.

First of all, as we said we don't have an analytical formula for the variational objective (our loss). We can only access (unbiased) samples, hence the next question.

**Exercise:**
Try to sample using 100 different random seeds the ELBO with [2, 10, 100, 1000] MC samples and plot their distribution with boxplots.

In [ ]:
elbo_samples = pd.DataFrame()
n_repetition = 100

for Nmc in [2, 10, 100, 1000]:
    elbo_samples[Nmc] = np.stack(
        [
            elbo_fn(q_params, p_params, jax.random.key(i), X, y, Nmc=Nmc)[0]
            for i in range(n_repetition)
        ]
    )


fig, ax = plt.subplots(figsize=[8, 4])
sns.boxplot(data=elbo_samples, whis=np.inf)
ax.set_title("Samples of the MC estimate of the ELBO")
ax.set_xlabel("Number of MC samples")
ax.set_ylabel("ELBO")
ax.margins(0, 0.05)

plt.show()

We said that, in case of large datasets, this can be computationally challenging, due to the evaluation of the likelihood $N_\mathrm{MC}$ times.
But we know that the ELBO can be approximated even further using mini-batching.
Taking a random subset of data $\mathcal{B}$, the approximation becomes

$$
    \mathbb{E}_{q_{\theta}}\log p(\boldsymbol{y}|\boldsymbol{X}, \boldsymbol{w}) \approx \dfrac{1}{N_\mathrm{MC}} \frac{N}{|\mathcal{B}|}\sum_{\tilde{\boldsymbol{w}}_i\sim q_\theta} \sum_{\boldsymbol{X}_j, \boldsymbol{y}_j\sim\mathcal{B}} \log p(\boldsymbol{y}_j|\boldsymbol{X}_j, \tilde{\boldsymbol{w}}_i)
$$

This introduces even more variance in the estimate of the ELBO but it allows to scale to (virtually) any sized dataset.
You are free to extend this to our ELBO estimator, but for the simple example we are using today, this is not required.

# Optimization

Ok, now we can move to the optimization of the ELBO:
$$
\boldsymbol \theta_{t+1} = \boldsymbol \theta_t + \text{lr}\cdot\nabla_{\boldsymbol \theta} \mathcal{L}(\boldsymbol \theta_t)
$$
Below you have a simple function to implement this update

In [ ]:
def sgd_update(params, gradients, learning_rate=1e-3):
    """
    Perform a stochastic gradient ascent update on the parameters.
    Args:
        params: Parameters to be updated.
        gradients: Gradients of the parameters.
        learning_rate: Learning rate for the update.
    """
    updated_params = jax.tree.map(lambda p, g: p + learning_rate * g, params, gradients)
    return updated_params

And here the function to compute a single step of the gradient of the ELBO (compute the gradient of the ELBO and make the gradient update).

**Exercise:**
Complete the next function to compute the gradient of the ELBO.

In [ ]:
@jax.jit(static_argnames=("elbo_fn", "Nmc"))  # JIT compile for speed
def train_step(elbo_fn, q_params, p_params, rng, X, y, Nmc=1, learning_rate=1e-3):
    """
    Perform a single training step of the ELBO optimization.
    It computes the gradient of the ELBO and makes a gradient ascent update.

    Args:
        elbo_fn: Function to compute the ELBO.
        q_params: Parameters of the posterior distribution.
        p_params: Parameters of the prior distribution.
        rng: Random number generator key.
        X: Input data.
        y: Labels.
        Nmc: Number of Monte Carlo samples to use for the ELBO estimation.
        learning_rate: Learning rate for the gradient ascent update.
    """
    # @@ COMPLETE @@
    # q_params_grad, (likelihood, kl) =
    # q_params =
    return q_params, (likelihood, kl)

**Exercise:**
Write the training loop to optimize the ELBO (use a learning rate of $10^{-3}$). At every step, store the value of the ELBO, of the expected likelihood and the KL. Use 100 Monte Carlo samples to estimate the ELBO at every step. Run the optimization for 10,000 iterations.

In [ ]:
rng = jax.random.key(0)
q_params = GaussianDiagonal(jnp.zeros(2), jnp.zeros(2))

n_iterations = 10000
Nmc = 100
learning_rate = 1e-3

elbo_summary = []  # Keep track of the ELBO
lik_summary = []  # Keep track of the likelihood
kl_summary = []  # Keep track of the KL divergence

with tqdm(total=n_iterations, desc="Training ELBO") as pbar:
    for step in range(n_iterations):
        rng, rng2 = jax.random.split(rng)
        q_params, (likelihood, kl) = train_step(
            elbo_fn, q_params, p_params, rng2, X, y, Nmc, learning_rate=learning_rate
        )
        lik_summary.append(likelihood)
        kl_summary.append(kl)
        elbo_summary.append(likelihood - kl)
        pbar.update(1)
        if step % 500 == 0:
            pbar.set_postfix_str(
                f"ELBO={likelihood - kl:.2f}, KL={kl:.2f}, Lik={likelihood:.2f}"
            )

In [ ]:
print("Converged posterior")
print("Mean =", q_params.mean)
print("Scale =", np.exp(q_params.log_var / 2))

**Exercise:**
Assess convergence of the optimization by plotting the three metrics.

In [ ]:
fig, (ax0, ax1, ax2) = plt.subplots(1, 3, figsize=[8, 2.5])

ax0.plot(elbo_summary, label="ELBO")
ax1.plot(
    lik_summary,
    color="tab:orange",
    label=r"$E_{q(\mathbf{w})} \log p(\mathbf{y}|\mathbf{X},\mathbf{w})$",
)
ax2.plot(
    kl_summary,
    color="tab:green",
    label=r"$\mathrm{KL}[{q(\mathbf{w})}||{p(\mathbf{w})}]$",
)

ax0.semilogx()
ax1.semilogx()
ax2.semilogx()

ax0.axhline(-2.68, color="xkcd:red", label=r"$\log p(\mathbf{y}|\mathbf{X})$")
ax0.legend(bbox_to_anchor=(1.05, -0.05), loc="lower right")
ax1.legend(bbox_to_anchor=(1.05, -0.05), loc="lower right")
ax2.legend(bbox_to_anchor=(1.05, -0.05), loc="lower right")

ax0.set_xlabel("Iteration")
ax1.set_xlabel("Iteration")
ax2.set_xlabel("Iteration")

plt.show()

**Question:**
Analyze the behaviour of these three values and comment the plots. Focus you analysis on the breakdown of the ELBO in its parts.

## Making predictions
Now that we have trained the model, we can use it to make predictions on new data.


\begin{equation}
\mathbb{E}_{q(\boldsymbol{w})}h(\boldsymbol{w}^\top\boldsymbol{x}_\mathrm{new}) = \int h(\boldsymbol{w}^\top\boldsymbol{x}_\mathrm{new}) q(\boldsymbol{w}) \mathrm{d}\boldsymbol{w}
\end{equation}


With 1000 samples, compute the probability $P (y_\mathrm{new} = 1 | \boldsymbol{x}_\mathrm{new}, \boldsymbol{X}, \boldsymbol{y})$ when $\boldsymbol{x}_\mathrm{new} = [2,-4]^\top$. Compare the result with the number you got from the previous lab.


In [ ]:
@jax.jit(static_argnames=("Nmc", "sample_fn"))  # JIT compile for speed
def predict_y(sample_fn, q_params, Xt, rng, Nmc=10):
    """
    Compute the outputs of the model by sampling the posterior, then take the expectation

    Args:
        sample_fn: Function to sample from the posterior.
        q_params: Parameters of the posterior distribution.
        Xt: Input data for prediction.
        rng: Random number generator key.
        Nmc: Number of Monte Carlo samples to use for the prediction.
    """

    @jax.vmap
    def predict_y_single(rng):
        """
        Predict the output for a single sample of the posterior.

        Note: This function is vectorized to allow for multiple samples of the posterior.
        """
        # @@ COMPLETE @@
        # w =
        # yp =
        return yp

    rng = jax.random.split(rng, Nmc)
    # @@ COMPLETE @@
    ## Compute the predictions for all the samples
    # yp_samples =
    ## Compute the mean of the predictions
    # yp =
    return yp


x = jnp.array([2, -4])
yp = predict_y(sample_gaussian_diagonal, q_params, x, rng, 1000)
print(yp)

**Execise:**
Let's now plot the distribution at convergence and the predictions on a grid of points.

In [ ]:
import scipy.stats


def plot_gaussian(params, **kwargs):
    ax = kwargs.pop("ax", plt.gca())
    xx, yy, w_plot = get_grid(xlim=(-1.5, 5.5), N=250)
    if isinstance(params, GaussianDiagonal):
        cov = np.exp(params.log_var) * np.eye(2)
    else:
        cov = params.L @ params.L.T
    zz = (
        scipy.stats.multivariate_normal(mean=params.mean, cov=cov)
        .pdf(w_plot)
        .reshape(*xx.shape)
    )

    levels = np.linspace(1e-5, np.max(zz), 10)
    ax.contourf(xx, yy, zz, cmap="cividis", alpha=0.8, levels=levels)
    ax.contour(xx, yy, zz, cmap="cividis", levels=levels)

    ax.set_xlabel(r"$\boldsymbol{w}_0$")
    ax.set_ylabel(r"$\boldsymbol{w}_1$")


def plot_posterior(ax):
    xx, yy, w_plot = get_grid(xlim=(-1.5, 5.5), N=250)
    zz = np.zeros(len(w_plot))
    for i, w in enumerate(w_plot):
        zz[i] = bernoulli_logdensity(y, model(w, X)).sum() - 0.5 * w.T @ w
    zz = np.exp(zz).reshape(*xx.shape)
    levels = np.linspace(1e-5, np.max(zz), 10)
    ax.contourf(xx, yy, zz, cmap="cividis", alpha=0.8, levels=levels)
    ax.contour(xx, yy, zz, cmap="cividis", levels=levels)
    ax.set_xlabel(r"$\boldsymbol{w}_0$")
    ax.set_ylabel(r"$\boldsymbol{w}_1$")

In [ ]:
fig, (ax0, ax1, ax2) = plt.subplots(1, 3, figsize=[12, 4])

plot_gaussian(p_params, ax=ax0)
plot_posterior(ax=ax1)
plot_gaussian(q_params, ax=ax2)

ax0.set_title("Prior")
ax1.set_title("True posterior")
ax2.set_title("Variational approx.")

plt.show()

In [ ]:
def plot_decision_boundary(xx, yy, P, ax):
    P = P.reshape(*xx.shape)
    levels = [0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 1]
    cs = ax.contour(xx, yy, P, levels, colors="k", linewidths=1.8, zorder=100)
    ax.clabel(cs, inline=1, fontsize=10)
    cs = ax.contourf(xx, yy, P, levels, cmap="Purples_r", alpha=0.5)


xx, yy, Xt = get_grid((-7, 7), N=50)
ps = predict_y(sample_gaussian_diagonal, q_params, Xt, rng, Nmc=1000)

fig, ax = plt.subplots(figsize=[5, 4])
plot_decision_boundary(xx, yy, ps, ax=ax)
plot_data(X, y, ax=ax)

ax.set_xlabel(r"$\boldsymbol{x}_0$")
ax.set_ylabel(r"$\boldsymbol{x}_1$")
ax.set_title("Predictive density")
plt.show()

**Exercise:**
At convergence, sample 1000 times the ELBO like we did before, and plot it with a boxplot.

In [ ]:
converged_elbos = pd.DataFrame()
n_repetition = 1000
converged_elbos["Diagonal posterior"] = np.stack(
    [
        elbo_fn(q_params, p_params, jax.random.key(i), X, y, Nmc=1000)[0]
        for i in range(n_repetition)
    ]
)

fig, ax = plt.subplots(figsize=[4, 3])
sns.boxplot(
    data=converged_elbos,
    whis=np.inf,
)
ax.axhline(-2.68, color="xkcd:red", label=r"$p(\boldsymbol{y}|\boldsymbol{X})$")
ax.set_title("ELBO at convergence", y=1.02)
ax.legend()
plt.show()

# 6. Alternatives to Mean Field Variational Inference

As we saw from one of the previous question, the approximation that we used is very rough: it recoves some properties of the true posterior but fails to capture the strong correlation that exists in the parameter space.
What we need to do it to increase the complexity and the expressiveness of the variational posterior.
The first step that we can do is to introduce a non-diagonal covariance $\boldsymbol{w} \sim \mathcal{N}(\mu, \Sigma)$, where the covariance $\Sigma=LL^\top$ ($L$ is a lower triangular matrix).

Sampling from such distribution is possible again using the reparameterization trick,

\begin{equation}
\tilde{\boldsymbol{w}} = \mu + \mathrm{Tril}(L)\boldsymbol{\varepsilon} \quad \boldsymbol{\varepsilon} \sim \mathcal{N}(0, \mathrm{I})
\end{equation}

where $\mathrm{Tril}(\cdot)$ returns the lower triangular part of the matrix (the other elements are set to 0).

**Exercise:**
Complete the next functions to model a full covariance Gaussian distribution.

In [ ]:
class GaussianFullCov(ParametrizedDistribution):
    mean: jnp.array
    L: jnp.array


def sample_gaussian_fullcov(key, params: GaussianFullCov):
    """
    Sample from a Gaussian distribution with full covariance.

    Args:
        key: Random number generator key.
        params: A named tuple containing the mean and Cholesky factor of the Gaussian.
    """
    # @@ COMPLETE @@
    # eps =
    # out =
    return out

**Exercise:** Try to sample from the distribution and print the result.

In [ ]:
q_params = GaussianFullCov(jnp.zeros(2), jnp.eye(2))

print(sample_gaussian_fullcov(jax.random.key(1), q_params))

Now we need to compute the KL divergence between $q$ Gaussian with full covariance and $p$ Gaussian with diagonal covariance.
Remember that the expression of the KL divergence between multivariate Gaussians $q = \mathcal{N}(\boldsymbol{\mu}_q, \Sigma_q)$ and $p = \mathcal{N}(\boldsymbol{\mu}_p, \Sigma_p)$ is as follows:

\begin{equation}
\mathrm{KL}[q || p] =
\frac{1}{2} \mathrm{Tr}(\Sigma_p^{-1} \Sigma_q)
+ \frac{1}{2} (\mu_p - \mu_q)^{\top} \Sigma_p^{-1} (\mu_p - \mu_q)
- \frac{D}{2}
+ \frac{1}{2} \log\left( \frac{\mathrm{det}\Sigma_p}{\mathrm{det}\Sigma_q} \right)
\end{equation}

**Question:**
Given that $q = \mathcal{N}(\boldsymbol{\mu}_q, LL^\top)$ and $p = \mathcal{N}(\boldsymbol{\mu}_p, \sigma^2_p\mathrm{I})$, write the simplified KL divergence.

*Hints:*

- $\mathrm{Tr}(\boldsymbol L \boldsymbol L^\top) = \sum_i\mathrm{diag}(\boldsymbol L)_i^2$

- $\log\mathrm{det}(\boldsymbol L\boldsymbol L^\top) = \sum_i\log\mathrm{diag}(\boldsymbol L)_i^2$


Below, you'll find the KL implemented.

In [ ]:
def kl_full_diag(q_params: GaussianFullCov, p_params: GaussianDiagonal):
    """
    Compute the KL divergence between a Gaussian distribution with full covariance
    and a Gaussian distribution with diagonal covariance.
    Args:
        q_params: A named tuple containing the mean and Cholesky factor of the first Gaussian.
        p_params: A named tuple containing the mean and log variance of the second Gaussian.
    """
    # @@ COMPLETE @@
    # kl =
    return kl

In [ ]:

print(kl_full_diag(q_params, p_params))

**Exercise:**
Create the a new elbo to use Gaussian with full covariance. Because we wrote the code for computing the ELBO to be as modular as possible, you can simply change the KL divergence function and the sampling function and everything else will work.

In [ ]:
# @@ COMPLETE @@
elbo_fn = create_elbo_fn(
)

**Exercise:**
Train this new model exactly the same as before

In [ ]:
rng = jax.random.key(0)
q_params = GaussianFullCov(jnp.zeros(2), jnp.eye(2))

n_iterations = 10000
Nmc = 100
learning_rate = 1e-3

elbo_summary = []
lik_summary = []
kl_summary = []

with tqdm(total=n_iterations, desc="Training ELBO") as pbar:
    for step in range(n_iterations):
        rng, rng2 = jax.random.split(rng)
        q_params, (likelihood, kl) = train_step(
            elbo_fn, q_params, p_params, rng2, X, y, Nmc, learning_rate=learning_rate
        )
        lik_summary.append(likelihood)
        kl_summary.append(kl)
        elbo_summary.append(likelihood - kl)
        pbar.update(1)
        if step % 500 == 0:
            pbar.set_postfix_str(
                f"ELBO={likelihood - kl:.2f}, KL={kl:.2f}, Lik={likelihood:.2f}"
            )

In [ ]:
print("Converged posterior")
print("Mean =", q_params.mean)
print("Scale matrix L =\n", q_params.L)
print("Covariance =", q_params.L @ q_params.L.T)

**Exercise:**
Plot the train curves

In [ ]:
fig, (ax0, ax1, ax2) = plt.subplots(1, 3, figsize=[8, 2.5])

ax0.plot(elbo_summary, label="ELBO")
ax1.plot(
    lik_summary,
    color="tab:orange",
    label=r"$E_{q(\boldsymbol{w})} \log p(\boldsymbol{y}|\boldsymbol{X},\boldsymbol{w})$",
)
ax2.plot(
    kl_summary,
    color="tab:green",
    label=r"$\mathrm{KL}[{q(\boldsymbol{w})}||{p(\boldsymbol{w})}]$",
)

ax0.semilogx()
ax1.semilogx()
ax2.semilogx()

ax0.axhline(-2.68, color="xkcd:red", label=r"$\log p(\boldsymbol{y}|\boldsymbol{X})$")
ax0.legend(bbox_to_anchor=(1.05, -0.05), loc="lower right")
ax1.legend(bbox_to_anchor=(1.05, -0.05), loc="lower right")
ax2.legend(bbox_to_anchor=(1.05, -0.05), loc="lower right")

ax0.set_xlabel("Iteration")
ax1.set_xlabel("Iteration")
ax2.set_xlabel("Iteration")

plt.show()

**Exercise:**
Plot the densities.

In [ ]:
fig, (ax0, ax1, ax2) = plt.subplots(1, 3, figsize=[12, 4])

plot_gaussian(p_params, ax=ax0)
plot_posterior(ax=ax1)
plot_gaussian(q_params, ax=ax2)

ax0.set_title("Prior")
ax1.set_title("True posterior")
ax2.set_title("Variational approx.")

plt.show()

**Exercise:**
Plot the predictions.

In [ ]:
xx, yy, Xt = get_grid((-7, 7), N=50)
ps = predict_y(sample_gaussian_fullcov, q_params, Xt, rng, Nmc=1000)

fig, ax = plt.subplots(figsize=[5, 4])
plot_decision_boundary(xx, yy, ps, ax=ax)
plot_data(X, y, ax=ax)

ax.set_xlabel(r"$\boldsymbol{x}_0$")
ax.set_ylabel(r"$\boldsymbol{x}_1$")
ax.set_title("Predictive density")
plt.show()

**Exercise:**
Plot the ELBO at convergence.

In [ ]:
converged_elbos["Full cov. posterior"] = np.stack(
    [
        elbo_fn(q_params, p_params, jax.random.key(i), X, y, Nmc=1000)[0]
        for i in range(n_repetition)
    ]
)
fig, ax = plt.subplots(figsize=[4, 3])
sns.boxplot(data=converged_elbos, whis=np.inf)
ax.axhline(-2.68, color="xkcd:red", label=r"$p(\boldsymbol{y}|\boldsymbol{X})$")
ax.set_title("ELBO at convergence", y=1.02)
ax.legend()
plt.show()

**Question:**
Do you observe something interesting?
Remember what the ELBO represents. It is the lower bound of the marginal distribution $p(\boldsymbol{y}|\boldsymbol{X})$. Check the first lab on Bayesian linear regression if you don't remember what the marginal distribution measures. Based solely on this value, which model would you choose? Why?

# Using NumPyro

Finally, like we did for MCMC, we can use NumPyro to perform Variational Inference.
Refer to the MCMC lab for a gentle introduction to NumPyro.
Here, we will directly jump to the Variational Inference part.
First, we need to define the model.

In [ ]:
import numpyro
import numpyro.distributions as dist


def model(X, y):
    N, D = X.shape
    # Prior on model parameters
    w = numpyro.sample("w", dist.Normal(jnp.zeros(D), jnp.ones(D)))
    # Deterministic transformation to get the probabilities
    probs = logistic(X @ w)

    # Likelihood with i.i.d. Bernoulli observations
    with numpyro.plate("data", N):
        numpyro.sample(
            "obs", dist.Bernoulli(probs=probs), obs=y
        )  # <-- Note the obs= argument

Now we can directly use NumPyro to perform variational inference. The variational posterior is called `guide` in NumPyro.

In [ ]:
from numpyro.infer import SVI, Trace_ELBO
from numpyro.infer.autoguide import AutoDiagonalNormal, AutoMultivariateNormal

# Choose the guide (i.e. the variational posterior)
# AutoDiagonalNormal -> Mean Field VI
# AutoMultivariateNormal -> Full Covariance VI
guide = AutoDiagonalNormal(model)
# guide = AutoMultivariateNormal(model)

elbo_fn = Trace_ELBO(num_particles=100)
#                    ^^^^^^^^^^^^^^^^^  This is what we called Nmc before

optim = numpyro.optim.SGD(1e-3)  # Optimizer (same as before)

# Create the SVI object
svi = SVI(model, guide, optim, elbo_fn)

# Run the optimization
svi_result = svi.run(jax.random.key(0), 10000, X, y)

In [ ]:
svi_result.params  # Optimized parameters

If you did everything correctly, you should obtain results very similar to what you got before.
Try to run both the Mean Field and Full Covariance versions and compare the results with what you got before.
In both cases, you should obtain very similar results.

Let's double check the results computing the predictive distribution at out usual test point $\boldsymbol{x}_\mathrm{new} = [2, -4]^\top$.

In [ ]:
from numpyro.infer import Predictive

# This is how to create the predictive distribution, using the model, the guide and the optimized parameters
predictive = Predictive(model, guide=guide, params=svi_result.params, num_samples=1000)

# New test point
x_new = jnp.array([[2.0, -4.0]])
# Compute the predictions
predictions = predictive(jax.random.key(1), x_new, None)["obs"]

# Compute the mean prediction
mean_prediction = jnp.mean(predictions, axis=0)
print("P(y_new = 1 | x_new, X, y) =", mean_prediction)

You should again obtain results very similar to what you got before.

# Concluding remarks

In this lab, we saw how to implement Variational Inference from scratch using JAX.
We saw how to use the reparameterization trick to sample from the variational posterior and how to compute the KL divergence between two Gaussian distributions.
We then implemented the ELBO and its optimization using simple SGD.

Finally, we saw how to use NumPyro to perform Variational Inference in a much simpler way. While using NumPyro is definitely easier and faster, implementing VI from scratch is a very useful exercise to understand the underlying mechanics of the algorithm.